# Prepare the chest X-ray dataset for a CNN

This notebook downloads and extracts the Kaggle dataset, keeps its official test set, makes a reproducible stratified train/validation split, creates a stratified `train`, `val`, and `test` layout, checks exact duplicates without deleting images, writes a SHA-256 manifest, and displays labeled samples. Existing prepared files are verified and preserved rather than overwritten.

The Kaggle URL in the request appears shortened; this notebook uses the canonical dataset slug `paultimothymooney/chest-xray-pneumonia`. Kaggle may require you to sign in or configure your Kaggle API credentials before downloading. The image-copy step needs extra disk space (roughly another copy of the dataset).

## Download and extract

`kagglehub.dataset_download` downloads and extracts the dataset, then returns its local path. This cell looks for the expected `train` and `test` folders, including when they are nested under `chest_xray`.

In [ ]:
from pathlib import Path
import kagglehub

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from this project or set PROJECT_ROOT explicitly.")

DATASET_SLUG = "paultimothymooney/chest-xray-pneumonia"
SEED = 42
SPLIT_OUTPUT_DIR = None  # Set a new empty directory to create a separate split version.

download_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
print(f"Downloaded/extracted dataset to: {download_dir}")
print(f"Project directory: {PROJECT_ROOT}")

candidates = [download_dir]
candidates.extend(path for path in download_dir.rglob("*") if path.is_dir())
data_root = next(
    (
        path
        for path in candidates
        if (path / "train").is_dir() and (path / "test").is_dir()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError(
        f"Could not find folders named 'train' and 'test' under {download_dir}. "
        "Check that the Kaggle dataset downloaded and extracted correctly."
    )

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
class_names = sorted(
    path.name for path in (data_root / "train").iterdir() if path.is_dir()
)
if not class_names:
    raise FileNotFoundError(f"No class folders found in {data_root / 'train'}")

print(f"Data root: {data_root}")
print(f"Classes: {class_names}")

## Make stratified splits and copy images

The dataset's official `test` folder stays untouched. The notebook combines the original `train` and (if present) `val` images, then stratifies a validation subset by class. The source test set is already close to 10% of this dataset, so the resulting proportions are approximately 80/10/10. Re-running this cell with the same seed produces the same split.

In [ ]:
import hashlib
import shutil
from collections import Counter
from sklearn.model_selection import train_test_split

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as image_file:
        for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def collect_images(split_names):
    records = []
    for split_name in split_names:
        split_dir = data_root / split_name
        if not split_dir.is_dir():
            continue
        for class_name in class_names:
            class_dir = split_dir / class_name
            if class_dir.is_dir():
                records.extend(
                    (image_path, class_name)
                    for image_path in sorted(class_dir.rglob("*"))
                    if image_path.is_file()
                    and image_path.suffix.lower() in IMAGE_EXTENSIONS
                )
    return records

training_pool = collect_images(["train", "val"])
test_records = collect_images(["test"])
if not training_pool or not test_records:
    raise FileNotFoundError(
        "No images were found in the expected class folders. "
        f"Training images: {len(training_pool)}, test images: {len(test_records)}."
    )

train_records, val_records = train_test_split(
    training_pool,
    test_size=0.10,
    random_state=SEED,
    stratify=[label for _, label in training_pool],
)
splits = {"train": train_records, "val": val_records, "test": test_records}
output_dir = Path(SPLIT_OUTPUT_DIR) if SPLIT_OUTPUT_DIR is not None else download_dir.parent / "chest_xray_cnn_splits"

if output_dir.exists():
    missing_splits = [name for name in splits if not (output_dir / name).is_dir()]
    if missing_splits:
        raise FileExistsError(
            f"Existing split {output_dir} is incomplete (missing {missing_splits}). "
            "It was not changed; choose a new empty SPLIT_OUTPUT_DIR after review."
        )
    expected_hashes = {
        split_name: {
            (class_name, sha256_file(source_path))
            for source_path, class_name in records
        }
        for split_name, records in splits.items()
    }
    actual_hashes = {
        split_name: {
            (class_dir.name, sha256_file(image_path))
            for class_dir in (output_dir / split_name).iterdir()
            if class_dir.is_dir()
            for image_path in class_dir.rglob("*")
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
        }
        for split_name in splits
    }
    if actual_hashes != expected_hashes:
        raise FileExistsError(
            f"Existing split {output_dir} differs from the requested source split. "
            "It was not changed. Set SPLIT_OUTPUT_DIR to a new empty directory to create a separate version."
        )
    print(f"Existing split matches this deterministic assignment; preserving files: {output_dir}")
else:
    for split_name, records in splits.items():
        for source_path, class_name in records:
            destination_dir = output_dir / split_name / class_name
            destination_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_dir / source_path.name)
    print(f"Created split folders: {output_dir}")

for split_name, records in splits.items():
    counts = Counter(label for _, label in records)
    print(f"{split_name:>5}: {len(records):,} source images | " + ", ".join(
        f"{name}: {counts.get(name, 0):,}" for name in class_names
    ))


## Check duplicate images and write the manifest

This checks the generated split folders for byte-identical image files using SHA-256, including duplicates spanning splits. It never deletes or changes existing images. If duplicates are found, the notebook stops so the split can be reviewed and versioned explicitly. The final manifest records paths, labels, hashes, image dimensions, image modes, class counts, and the random seed.

In [ ]:
from collections import defaultdict
import csv
import io
import json
import subprocess
import sys
from PIL import Image

hash_to_files = defaultdict(list)
for split_name in ("train", "val", "test"):
    split_dir = output_dir / split_name
    if not split_dir.is_dir():
        continue
    for image_path in sorted(split_dir.rglob("*")):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            hash_to_files[sha256_file(image_path)].append(image_path)

if not hash_to_files:
    raise FileNotFoundError(f"No images found in split folders under {output_dir}")

duplicate_groups = [paths for paths in hash_to_files.values() if len(paths) > 1]
cross_split_groups = [
    paths for paths in duplicate_groups
    if len({path.relative_to(output_dir).parts[0] for path in paths}) > 1
]
print(f"Unique image files checked: {sum(map(len, hash_to_files.values())):,}")
print(f"Exact duplicate groups: {len(duplicate_groups):,}")
print(f"Duplicate groups spanning splits: {len(cross_split_groups):,}")
if duplicate_groups:
    for paths in duplicate_groups[:20]:
        print("Duplicate group:", *(str(path.relative_to(output_dir)) for path in paths), sep="\n  ")
    raise ValueError(
        "Exact duplicate images were found. No images were deleted; review the reported groups "
        "and create a separately documented split before continuing."
    )

class_mapping = {name: index for index, name in enumerate(class_names)}
manifest_rows = []
image_modes = Counter()
widths, heights = [], []
for split_name in ("train", "val", "test"):
    for class_name in class_names:
        for image_path in sorted((output_dir / split_name / class_name).rglob("*")):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                width, height = image.size
                mode = image.mode
            image_modes[mode] += 1
            widths.append(width)
            heights.append(height)
            manifest_rows.append({
                "relative_path": image_path.relative_to(output_dir).as_posix(),
                "split": split_name,
                "label": class_name,
                "class_id": class_mapping[class_name],
                "sha256": sha256_file(image_path),
                "width": width,
                "height": height,
                "mode": mode,
            })
manifest_rows.sort(key=lambda row: row["relative_path"])
manifest_path = PROJECT_ROOT / "data" / "dataset_manifest.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_buffer = io.StringIO(newline="")
writer = csv.DictWriter(manifest_buffer, fieldnames=[
    "relative_path", "split", "label", "class_id", "sha256", "width", "height", "mode"
])
writer.writeheader()
writer.writerows(manifest_rows)
manifest_contents = manifest_buffer.getvalue()
manifest_hash = hashlib.sha256(manifest_contents.encode("utf-8")).hexdigest()
if manifest_path.exists():
    if sha256_file(manifest_path) != manifest_hash:
        raise FileExistsError(
            f"A different canonical manifest already exists at {manifest_path}. "
            "It was not overwritten; review and version the new split explicitly."
        )
else:
    manifest_path.write_text(manifest_contents, encoding="utf-8")
manifest_counts = {
    split: {name: sum(row["split"] == split and row["label"] == name for row in manifest_rows)
            for name in class_names}
    for split in ("train", "val", "test")
}
summary = {
    "dataset": DATASET_SLUG,
    "manifest": "data/dataset_manifest.csv",
    "manifest_sha256": manifest_hash,
    "class_names": class_names,
    "class_mapping": class_mapping,
    "splits": ["train", "val", "test"],
    "split_seed": SEED,
    "split_method": "Sorted source paths; stratified 10% validation split from source train+val; source test retained; exact duplicates checked but never deleted.",
    "counts": manifest_counts,
    "total_images": len(manifest_rows),
    "unique_sha256": len({row["sha256"] for row in manifest_rows}),
    "duplicate_groups": 0,
    "unreadable_images": 0,
    "source_image_pixel_data_in_repository": False,
    "image_modes": dict(image_modes),
    "dimensions": {"width_min": min(widths), "width_max": max(widths),
                   "height_min": min(heights), "height_max": max(heights)},
}
summary_path = PROJECT_ROOT / "data" / "dataset_manifest_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "data" / "validate_dataset.py"), "--split-dir", str(output_dir)],
    check=True,
)
print(f"Manifest: {manifest_path} (SHA-256 {manifest_hash})")


## Print and display sample images

The output includes each sample's class and filename, followed by a small image grid.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

sample_records = random.Random(SEED).sample(
    train_records, k=min(6, len(train_records))
)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax in axes.flat:
    ax.axis("off")

for ax, (image_path, label) in zip(axes.flat, sample_records):
    print(f"Class: {label:10} | File: {image_path.name}")
    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"), cmap="gray")
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Load the folders for a CNN

TensorFlow is included in the install cell near the top of this notebook. If `import tensorflow` previously failed, run that install cell using **Run Cell** (not a terminal command), wait for it to finish, and restart the notebook kernel if VS Code prompts you. Make sure the notebook is using the same Python environment where the install ran. The following creates Keras datasets; it does not train a model.

In [ ]:
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "val",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    output_dir / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Class order:", train_ds.class_names)